# SMILES 2026 Method 2 Binary Ablation Colab Runner

This notebook syncs the GitHub repo into a Drive-backed workspace, installs dependencies, and runs the ICR Probe binary-classifier ablation on `data/dataset.csv` with `Qwen/Qwen2.5-0.5B`.

Compared classifiers:

- logistic regression
- 2-hidden-layer MLP with `dropout=0.3` and `L2`
- 2-hidden-layer MLP with `dropout=0.3` and `L1 + L2`

Default ICR feature settings:

- keep all heads, mean-pool across heads
- `top_k = 10`
- z-score normalization before softmax
- truncation: disabled, full tokenized `prompt + response`


In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)


In [ ]:
import os
import subprocess
from pathlib import Path

TARGET_FOLDER = Path('/content/drive/MyDrive/hallucination_detection')
REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_draft.git'
REPO_NAME = 'hallucination_detection_draft'
REPO_PATH = TARGET_FOLDER / REPO_NAME
AUTO_STASH = True

TARGET_FOLDER.mkdir(parents=True, exist_ok=True)

def run(cmd: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result

print('TARGET_FOLDER =', TARGET_FOLDER)
print('REPO_PATH =', REPO_PATH)


In [ ]:
if not REPO_PATH.exists():
    run(f'git clone {REPO_URL}', cwd=TARGET_FOLDER)

run('git remote -v', cwd=REPO_PATH)
run('git branch --show-current', cwd=REPO_PATH)
run('git fetch origin', cwd=REPO_PATH)
status = run('git status --short', cwd=REPO_PATH, check=False).stdout.strip()

if status:
    print('Local changes detected.')
    if AUTO_STASH:
        run('git stash push -u -m "colab-auto-stash"', cwd=REPO_PATH)
    else:
        raise RuntimeError('Repo is dirty. Set AUTO_STASH = True or clean it manually.')

run('git pull --ff-only origin main', cwd=REPO_PATH)
run('git log --oneline -1', cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('cwd =', Path.cwd())


In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)


In [ ]:
import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Switch Colab runtime to GPU.')


## Binary-Classifier Ablation

The ablation evaluates the same fixed 24-dimensional ICR feature vector with three binary classifiers. The results are ranked by **mean validation accuracy**, with AUROC used as the next tie-breaker.

Outputs are written to `method2_icr_probe/artifacts/ablation/` inside the Drive-backed repo.


In [ ]:
run(
    'python method2_icr_probe/run_binary_ablation.py '
    '--batch-size 1 '
    '--cache-dtype float32',
    cwd=REPO_PATH,
)


In [ ]:
import json
import pandas as pd

ablation_dir = REPO_PATH / 'method2_icr_probe' / 'artifacts' / 'ablation'
leaderboard = pd.read_csv(ablation_dir / 'ablation_results.csv')
display(
    leaderboard[
        [
            'name',
            'classifier',
            'feature_dim',
            'hidden_dims',
            'regularization',
            'top_k',
            'z_normalize',
            'mean_val_accuracy',
            'mean_val_auroc',
            'mean_test_accuracy',
            'mean_test_auroc',
        ]
    ].sort_values(['mean_val_accuracy', 'mean_val_auroc'], ascending=[False, False])
)

best_payload = json.loads((ablation_dir / 'best_config.json').read_text())
print(json.dumps(best_payload, indent=2))
